# Generación del modelo

## Pasar los jsonl a datos en csv para poder cargarlos en la base de datos de la API

In [1]:
import pandas as pd
import duckdb
import json

In [2]:
# reviews_file = 'Kindle_Store.jsonl'
# reviews = pd.read_json(reviews_file, lines=True, chunksize=10)[['rating', 'title', 'parent_asin', 'user_id']]
# for chunk in reviews:
#     first_review = chunk.iloc[1]
#     print(first_review['rating'], first_review['title'], first_review['parent_asin'], first_review['user_id'])
#     break

In [3]:
# reviews_file = 'meta_Kindle_Store.jsonl'
# books = pd.read_json(reviews_file, lines=True, chunksize=10)
# for chunk in books:
#     first_book = chunk.iloc[1]
#     # print(first_book)
#     # break
#     print(first_book['title'], first_book['subtitle'], first_book['parent_asin'], first_book['categories'], first_book['images'], first_book['description'], first_book['details'])
#     break

In [4]:
con = duckdb.connect('amazon_etl.db')

### Obtener libros con más de 20 reseñas

In [5]:
con.execute("""
    DROP TABLE IF EXISTS popular_books;
    CREATE TEMP TABLE popular_books AS
    SELECT parent_asin
    FROM read_json_auto('Kindle_Store.jsonl')
    GROUP BY parent_asin
    HAVING COUNT(*) >= 500;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
total_books = con.execute("""
    SELECT COUNT(*) as total_books
    FROM popular_books
""").df()

total_books

,total_books
0,4601


### Encontrar usuarios con al menos 15 reseñas en total dentro de esos libros

In [7]:
con.execute("""
    DROP TABLE IF EXISTS active_users;
    CREATE TEMP TABLE active_users AS
    SELECT user_id
    FROM read_json_auto('Kindle_Store.jsonl')
    WHERE parent_asin IN (SELECT parent_asin FROM popular_books)
    GROUP BY user_id
    HAVING COUNT(*) BETWEEN 35 AND 50;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [8]:
total_users = con.execute("""
    SELECT COUNT(*) as total_users
    FROM active_users
""").df()

total_users

,total_users
0,2709


### Obtener sólo las reviews de los libros y usuarios filtrados

In [9]:
con.execute("""
    DROP TABLE IF EXISTS final_reviews;
    CREATE TEMP TABLE final_reviews AS
    SELECT user_id, parent_asin as item_id, rating, timestamp
    FROM read_json_auto('Kindle_Store.jsonl')
    WHERE parent_asin IN (SELECT parent_asin FROM popular_books)
    AND user_id IN (SELECT user_id FROM active_users);
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [10]:
total_reviews = con.execute("""
SELECT COUNT(*) as total_reviews
FROM final_reviews
""").df()

total_reviews

,total_reviews
0,111005


### Obtener metadatos de los libros seleccionados

In [11]:
con.execute("""
    CREATE TEMP TABLE final_meta AS
    SELECT DISTINCT parent_asin as item_id, title, subtitle, description, categories, images
    FROM read_json_auto('meta_Kindle_Store.jsonl')
    WHERE parent_asin IN (SELECT parent_asin FROM popular_books);
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [12]:
total_books = con.execute("""
SELECT COUNT(*) as total_books
FROM final_meta
""").df()

total_books

,total_books
0,4601


In [13]:
meta_sample = con.execute("""
    select * from final_meta
    limit 5;
""").df()

print(meta_sample.iloc[0])

item_id                                               B094XQFL8P
title                               Runaway (Empire High Book 5)
subtitle                                          Kindle Edition
description    [About the Author, Ivy Smoak is the internatio...
categories     [Kindle Store, Kindle eBooks, Literature & Fic...
images         [{'large': 'https://m.media-amazon.com/images/...
Name: 0, dtype: object


In [14]:
reviews_sample = con.execute("""
    select * from final_reviews
    limit 5;
""").df()

print(reviews_sample.iloc[0])

user_id      AEQ73HDUVHN2BTGZFG7YI3D6BODQ
item_id                        B088Z23LQ5
rating                                4.0
timestamp                   1676368219835
Name: 0, dtype: object


### Exportar a CSV

In [15]:
con.execute("COPY final_reviews TO 'kindle_reviews_seed.csv' (HEADER, DELIMITER ',')")
con.execute("COPY final_meta TO 'kindle_meta_seed.csv' (HEADER, DELIMITER ',')")

In [16]:
meta_df = pd.read_csv('kindle_meta_seed.csv')

In [17]:
print(meta_df.iloc[0]['categories'])

[Kindle Store, Kindle eBooks, Literature & Fiction]


In [18]:
rows_with_unwanted_categories = meta_df[meta_df['categories'].str.contains('Kindle', na=False)]
rows_with_unwanted_categories.count()

item_id        4601
title          4601
subtitle       4214
description    4601
categories     4601
images         4601
dtype: int64

### Mantener sólo las categorías relevantes

In [19]:
unwanted_categories = ['Kindle Store', 'Kindle eBooks', 'Kindle Unlimited', 'Kindle Short Reads', 'Kindle eTextbooks', 'Kindle Games & Active Content']
for category in unwanted_categories:
    meta_df['categories'] = meta_df['categories'].apply(lambda x: x.replace(category + ',', '') if pd.notnull(x) else x)

meta_df['categories'] = meta_df['categories'].apply(lambda x: x.replace('Kindle eBooks', '') if pd.notnull(x) else x)

meta_df['categories'] = meta_df['categories'].apply(lambda x: x.strip() if pd.notnull(x) else x)

In [20]:
rows_with_unwanted_categories = meta_df[meta_df['categories'].str.contains('Kindle', na=False)]
rows_with_unwanted_categories.count()

item_id        0
title          0
subtitle       0
description    0
categories     0
images         0
dtype: int64

In [21]:
meta_df.head()

,item_id,title,subtitle,description,categories,images
0,B094XQFL8P,Runaway (Empire High Book 5),Kindle Edition,"[About the Author, 'Ivy Smoak is the internati...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...
1,B075WQK8ZJ,Unsheltered: A Novel,Kindle Edition,"[Amazon.com Review, 'An Amazon Best Book of Oc...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...
2,B09JS55FW3,Reaper (Cradle Book 10),Kindle Edition,"[About the Author, Will Wight, 'lives in Flori...",[ Science Fiction & Fantasy],[{'large': 'https://m.media-amazon.com/images/...
3,B01JJ5CURW,To-Do List Formula: A Stress-Free Guide To Cre...,Kindle Edition,[],[ Business & Money],[{'large': 'https://m.media-amazon.com/images/...
4,B00KQBTC3O,Shopping for a Billionaire 1 (Shopping for a B...,Kindle Edition,"[From the Author, 'Please sign up for my New R...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...


### Exportar a csv

In [22]:
meta_df.to_csv('kindle_meta_cleaned.csv', index=False)
meta_df.head(100).to_csv('kindle_meta_cleaned_sample.csv', index=False)

In [23]:
len(meta_df)

4601

In [24]:
reviews_df = pd.read_csv('kindle_reviews_seed.csv')

In [25]:
reviews_df.head(100).to_csv('kindle_reviews_sample.csv', index=False)

In [26]:
reviews_df['user_id'].unique()

<StringArray>
['AEQ73HDUVHN2BTGZFG7YI3D6BODQ', 'AH6JMVJUPKFNC25N6AQI7RFYHZPQ',
 'AF5NIV3P32BC6MTXRTJK4VYM7BKQ', 'AHZAATNH532PP7NBOBM3EVRZJNTQ',
 'AE67DUY3UJHC7UXFMGXJJRJQFXEQ', 'AHNRQPYEH4ORFNEEM7MNLM73VWTQ',
 'AH32YEBAAXOZP2GCW4YCBTWL6O4Q', 'AGM5XLAIVOJH7UDUNXEYDRAOBQZQ',
 'AF7WYMY5XA3LZ6ZYNKMICCIARC5A', 'AEVJLEHPIHCUM4NAKL25KA67NOFQ',
 ...
 'AEHG3CC7MREUANRBGSHRCPD67TGA', 'AERMA6OXOCVYBYQCI4AIF46OJGNA',
 'AF7A3ALOYGLJQEU4PTYGMMNG54UQ', 'AEE4VVA7GP4BJOCMDIGSBP3ML2XA',
 'AH2UQ2VE4ILGMMTC73RJLE6VLJMQ', 'AHVPK23LU3MZ4B2D4DTOUHUJTLLQ',
 'AGS3L6QXRPQUCSPJUUMU6OB6LFEQ', 'AGDJBU7AENWOHFDWJQID5M2FARVQ',
 'AGVSC2JFHE3RFEI7JCNMB5EDKINQ', 'AEHC5F4PFXFOZ4Q63W3WYJBIKLJA']
Length: 2709, dtype: str

In [27]:
reviews_df.head()

,user_id,item_id,rating,timestamp
0,AEQ73HDUVHN2BTGZFG7YI3D6BODQ,B088Z23LQ5,4.0,1676368219835
1,AEQ73HDUVHN2BTGZFG7YI3D6BODQ,B00U88YML2,5.0,1675642379912
2,AEQ73HDUVHN2BTGZFG7YI3D6BODQ,B09ZB5FLZ4,5.0,1674518937616
3,AEQ73HDUVHN2BTGZFG7YI3D6BODQ,B07KPFLD6Q,5.0,1669488205993
4,AEQ73HDUVHN2BTGZFG7YI3D6BODQ,B09MCY9Y34,5.0,1665983353937


In [64]:
len(reviews_df)

111005

## Entrenar modelo con Surprise

In [41]:
from surprise import Dataset, SVD, Reader, accuracy
from surprise.model_selection import train_test_split

In [29]:
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(reviews_df[['user_id', 'item_id', 'rating']], reader)

In [48]:
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

In [49]:
algo = SVD()
algo.fit(trainset)


In [50]:
predictions = algo.test(testset)

In [51]:
accuracy.rmse(predictions)

RMSE: 0.7607


0.7607130593338862

In [52]:
df_test = pd.DataFrame(testset, columns=['user_id', 'item_id', 'rating_real'])
df_test.to_csv('testset_oraculo.csv', index=False)

In [54]:
import joblib

In [55]:
joblib.dump(algo, 'svd_model.pkl')

['svd_model.pkl']

## Generar embeddings de los libros